# HUDM 5001 — Programming for Data Science
## Session 12 · Exception Handling, Unit Testing, and Data Visualization
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Week_12/12_Exceptions_Testing_Visualization.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**


> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.  

About this notebook:  
This notebook is adapted from Prof. Youmi Suk's materials for HUDM 5001, Programming for Data Science ([original repository](https://github.com/youmilab/hudm5001)).  
Wording in this notebook was refined with Claude. I am responsible for its accuracy.  

Version: Nov.23.2026 12:32 am 

Last week's `Student` class takes whatever you hand it. Call `ann.add_score("ninety")` and the word goes into her list without a complaint. Nothing goes wrong until you ask for her average, and then Python stops with an error inside `average()`, a method that has worked fine all along. Today has three parts. Part A is about errors: how to read the messages Python prints, how to stop bad input with a message of your own, and how to keep a program running when some of its input is bad. In Part B you write small tests that check whether a class does what its docstrings promise, starting with `Student`. Part C turns from code to pictures. You will make the four charts that answer most first questions about a dataset, and save them the way a report needs them. Your final project asks for at least two figures, worth 10 of its 60 points.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Read a traceback from the bottom up, and name the exceptions you meet most often in data work.
- Stop bad input early with `raise` and an error message that says what was expected.
- Handle errors you expect with `try` and `except`, and explain why an `except:` with no exception name is risky.
- Use `assert` to check an assumption about your data or your code.
- Write unit tests with `unittest`, including a test that checks that bad input raises an error.
- Choose test cases (a typical value, the boundaries, an empty input, bad input, an old bug), and write a test before the code it tests.
- Make a histogram, a bar chart, a boxplot, and a scatter plot with matplotlib, label them, and save them for a report.
- Make an interactive chart with Plotly Express, and choose a chart that fits your question.
- Ask an AI tool for a chart by naming the chart type, the columns, and the parts to change, and check the chart that comes back.

### ✅ Prerequisites
- Session 11: classes, `__init__`, methods, `self`, the `Student` class, the `Program` class from Exercises 2 and 5, and inheritance (`class GradStudent(Student):`).
- Session 10: `return` and `None`, and list comprehensions. Session 07: `zip()`. Session 06: `conn.close()`. Session 03: `with open(...)`. From the pandas sessions you need `.loc`, `groupby`, and `value_counts()`.
- Run the Setup cell to import the libraries and load the penguins.

### 🧩 Key Concepts
`exception`  `traceback`  `raise`  `try / except`  `else / finally`  `assert`  `unit test`  `unittest.TestCase`  `assertEqual`  `assertRaises`  `test case`  `boundary`  `regression test`  `test-driven development`  `Figure`  `Axes`  `histogram`  `bar chart`  `boxplot`  `scatter plot`  `legend`  `savefig`  `plotly.express`

### 📖 Reading
- The official Python tutorial, [Chapter 8: Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html), sections 8.1 to 8.4
- The [`unittest` documentation](https://docs.python.org/3/library/unittest.html), only the section "Basic example"
- McKinney, *Python for Data Analysis*, §9.1, a short primer on matplotlib

### 🗺️ Today's Agenda
Part A. When code meets bad input

- 2.1 Bad input and the traceback
- 2.2 Exceptions have names
- 2.3 Raising your own errors
- 2.4 Handling errors with `try` and `except`

Part B. Testing your code

- 2.5 `assert` and your first test
- 2.6 Unit tests with `unittest`
- 2.7 What to test, and testing first

Part C. Showing your data

- 2.8 A figure is an object
- 2.9 Four charts for four questions
- 2.10 Several charts in one figure, and saving for your report
- 2.11 Interactive charts, and choosing a chart
- 2.12 Asking AI for a chart, and checking the result

---
## ▶️ Setup — run this cell first
This imports the libraries and loads the Palmer penguins, the data you cleaned and queried in Assignment 04: 344 penguins of three species from three islands in Antarctica, with each bird's bill length and depth, flipper length, body mass, and sex. Part A uses them for a few error messages, and Part C draws every chart from them. The cell also defines `run_tests()`, a small helper that runs a group of unit tests and prints a report. You do not need to read its code. Section 2.6 shows how to use it.

In [ ]:
# === SETUP — run me first! ===
import sys
import unittest

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# The penguins, read straight from the public course repo (no downloads).
PENGUINS_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/penguins.csv"
penguins = pd.read_csv(PENGUINS_URL)

def run_tests(test_class):
    """Run every test method of one unittest.TestCase class and print a report."""
    suite = unittest.TestLoader().loadTestsFromTestCase(test_class)
    unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)

print("pandas", pd.__version__, "| matplotlib", matplotlib.__version__)
print(penguins.shape)
print("Setup complete ✅")

---
## 2. Content

### Part A. When code meets bad input

### 2.1 Bad input and the traceback
Here is the `Student` class as we left it at the end of section 2.7 last week. Run the cell to define it in this notebook.

In [ ]:
class Student:
    """A student with a name and a list of scores."""

    passing_score = 60

    def __init__(self, name, scores=None):
        self.name = name
        self.scores = [] if scores is None else scores

    def average(self):
        """Return the mean of this student's scores, or None if there are none yet."""
        if len(self.scores) == 0:
            return None
        return sum(self.scores) / len(self.scores)

    def add_score(self, score):
        """Add one score to this student's list. Returns nothing."""
        self.scores.append(score)

    def passed(self):
        """Return True if the average reaches the passing score, and False with no scores yet."""
        if self.average() is None:
            return False
        return self.average() >= self.passing_score

    def __repr__(self):
        """Return a short description, written like the code that creates the student."""
        return f"Student(name='{self.name}', scores={self.scores})"

The docstring of `add_score` promises to add one score. It never says what a score is, and the method does not check. Watch what happens when a score arrives as a word.

In [ ]:
ann = Student("Ann", [90, 85])
ann.add_score("ninety")      # no complaint: the word goes into the list
print(ann)

Now ask for Ann's average: remove the `#` in the next cell and run it.

In [ ]:
# ann.average()

Python stops and prints a **traceback**, its report of where it was when things went wrong. Read it from the bottom up.

1. The last line names the error and gives a message: `TypeError: unsupported operand type(s) for +: 'int' and 'str'`. `sum()` adds the scores one at a time, and when it reaches `"ninety"` it is asked to add a number and a word, which Python cannot do.
2. Each block above the last line is one step on the way there. The lowest block shows the line inside `average()` where the error happened, marked with an arrow. The block above it shows the line in your cell that called `average()`.

The traceback does not show the real mistake. That was `ann.add_score("ninety")`, one cell earlier, and it ran without a problem. A traceback shows where Python gave up, which can be a long way from where the mistake was made. (Jupyter and VS Code label each block a little differently from Colab, but the order is the same.) Put the `#` back so the notebook still runs from top to bottom.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/traceback_anatomy.png" width="800" alt="A Colab traceback for ann.average(), read from the bottom up: the last line names the TypeError, the arrow above it marks line 14 inside average(), and the top block shows the call in your cell. The real mistake, add_score with a word, ran earlier without an error.">
</p>

Figure source: Made with Claude.

A word in the list at least causes a crash. A typo that is still a number causes nothing at all.

In [ ]:
bob = Student("Bob", [72, 88])
bob.add_score(950)           # meant to type 95
print(bob.average())         # 370.0, and no error
print(bob.passed())          # True

An average of 370 on a 100-point scale is impossible, and Python has no way to know that. Python raises an error only when it cannot run a line. When a line runs fine but the value makes no sense for your data, Python stays quiet, and the wrong number travels on into your tables and charts. Section 2.3 shows how to make the class check its own input.

### 2.2 Exceptions have names
When Python cannot run a line, it raises an **exception**: it stops, creates an object that describes the problem, and prints that object as the last line of the traceback. The first word of that line is the exception's name, and the name tells you what kind of problem to look for. You have met most of these already.

| Exception | What it means | A line that raises it | Where you have seen it |
|---|---|---|---|
| `NameError` | Python does not know the name | `print(total_mass)`, before `total_mass` exists | running a cell before the cell that creates the variable |
| `KeyError` | no such key in a dictionary, or no such column in a DataFrame | `penguins["Body Mass"]` | a typo in a column name |
| `TypeError` | an operation that does not work for this type of value | `90 + "ninety"` | 2.1, and a method without `self` in Session 11 |
| `AttributeError` | the object has no attribute or method with that name | `penguins.average()` | forgetting `self.` in `__init__`, Session 11 |
| `ValueError` | the type is right, but the value cannot be used | `int("eighty")` | converting text to numbers |
| `IndexError` | a position past the end of a list | `[152, 124, 68][3]` | a loop that runs one step too far |
| `ZeroDivisionError` | division by zero | `sum([]) / len([])` | the average of an empty list, which is why `average()` checks for one |
| `FileNotFoundError` | there is no file at that path | `pd.read_csv("penguin.csv")` | a typo in a file name, or a file that is not in your Colab session |

An exception is an object, built from a class like any other object. You can even create one without raising it, and then it just sits there like any other value.

In [ ]:
problem = ValueError("score must be between 0 and 100, got 950")    # create an exception object
print(type(problem))       # its class
print(problem)             # its message

All of these classes inherit, directly or through a parent class, from one class named `Exception`, the way `GradStudent` inherited from `Student`. In 2.4 you will catch exceptions by their class.

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★☆☆☆☆  **Skills:** reading tracebacks, exception names

Each line in the next cell raises a different exception.  
(a) Before you run anything, write the exception you expect after `# my guess:` at the end of each line.  
(b) Remove the `#` at the start of one line, run the cell, and compare the last line of the traceback with your guess. Put the `#` back before you try the next line.  
(c) Which line raises the same exception as `ann.average()` in 2.1, and for the same reason?

In [ ]:
# penguins["Species"]                # my guess:
# int("3.5")                         # my guess:
# "Adelie" + 1                       # my guess:
# penguins["species"].mean_mass()    # my guess:
# [152, 124, 68][3]                  # my guess:
# print(chinstrap_count)             # my guess:

<details>
<summary>▶ Show solution</summary>

```python
# penguins["Species"]                # KeyError: 'Species'. Column names are case-sensitive, and the column is "species"
# int("3.5")                         # ValueError: invalid literal for int() with base 10: '3.5'. int() reads whole numbers only; float("3.5") works
# "Adelie" + 1                       # TypeError: can only concatenate str (not "int") to str
# penguins["species"].mean_mass()    # AttributeError: 'Series' object has no attribute 'mean_mass'
# [152, 124, 68][3]                  # IndexError: list index out of range. The positions are 0, 1, and 2
# print(chinstrap_count)             # NameError: name 'chinstrap_count' is not defined

# (c) "Adelie" + 1. Like sum() in 2.1, it asks Python to add a word and a number, so both raise a TypeError.
```

</details>

### 2.3 Raising your own errors
Python only checks what it needs in order to run a line. It cannot know that a score has to be a number from 0 to 100, so the class has to check that itself. The keyword `raise` does the job. It stops the method at once and sends an exception, with a message you write, back to the code that called the method.

In [ ]:
class Student:
    """A student with a name and a list of scores, each a number from 0 to 100."""

    passing_score = 60

    def __init__(self, name, scores=None):
        self.name = name
        self.scores = []
        if scores is not None:
            for score in scores:
                self.add_score(score)       # <---- starting scores go through the same checks

    def average(self):
        """Return the mean of this student's scores, or None if there are none yet."""
        if len(self.scores) == 0:
            return None
        return sum(self.scores) / len(self.scores)

    def add_score(self, score):
        """Add one score, a number from 0 to 100, to this student's list. Returns nothing."""
        if not isinstance(score, (int, float)):
            raise TypeError(f"score must be a number, got {repr(score)}")       # <---- not a number: stop here
        if score < 0 or score > 100:
            raise ValueError(f"score must be between 0 and 100, got {score}")   # <---- a number, but out of range
        self.scores.append(score)           # reached only when both checks pass

    def passed(self):
        """Return True if the average reaches the passing score, and False with no scores yet."""
        if self.average() is None:
            return False
        return self.average() >= self.passing_score

    def __repr__(self):
        """Return a short description, written like the code that creates the student."""
        return f"Student(name='{self.name}', scores={self.scores})"

ann = Student("Ann", [90, 85])
print(ann)
# ann.add_score("ninety")     # TypeError: score must be a number, got 'ninety'
# ann.add_score(950)          # ValueError: score must be between 0 and 100, got 950

Remove the `#` from one of the last two lines at a time and run the cell, then put the `#` back. Each mistake now stops the program at the moment it happens, in the line that made it, and the message says what the method expected and what it got. A few details in the new class are worth a closer look:

- `isinstance(score, (int, float))` is `True` when `score` is an `int` or a `float`. Given a tuple of classes, `isinstance` checks for any of them.
- `repr(score)` calls the `__repr__` you met last week. For a string it keeps the quotes, so the message shows `'ninety'` and you can tell the score arrived as text.
- A `TypeError` says the value has the wrong type. A `ValueError` says the type is fine but the value cannot be used. Pick the built-in exception that fits, because the code that calls yours catches errors by their class, as you will see in 2.4.
- `__init__` now sends every starting score through `add_score`, so `Student("Bob", [72, 950])` is refused too. With the checks in one method, a later change to the rules happens in one place.

Why stop the program instead of printing a warning and carrying on? A printed warning scrolls past and the bad score still goes into the list, while an exception keeps the bad value out.

Syntax:  
```python
def some_method(self, value):
    if value_is_wrong:                                         # <---- check first
        raise ValueError(f"what was expected, got {value}")    # <---- stop here and say what went wrong
    ...                                                        # <---- the normal work, reached only by good input
```

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★☆☆☆  **Skills:** `raise`, `ValueError`, error messages

The next cell has the `Program` class from Session 11. Its `enroll(n)` method accepts any number, so `m.enroll(-5)` lowers the enrollment, and `m.enroll(100)` squeezes 134 students into 40 seats.  
(a) Make `enroll(n)` raise a `ValueError` when `n` is less than 1. The message should include the value of `n`.  
(b) Make it raise a `ValueError` when enrolling `n` students would go over capacity. Check before you change `self.enrolled`, so that a refused enrollment leaves the program as it was. The message should say how many seats are left.  
(c) Remove the `#` from each test line, one at a time, and check that the right message appears. Then put the `#` back.

In [ ]:
class Program:
    """A degree program with an enrollment and a capacity."""

    def __init__(self, name, enrolled, capacity=40):
        self.name = name
        self.enrolled = enrolled
        self.capacity = capacity

    def seats_left(self):
        """Return how many seats are still open."""
        return self.capacity - self.enrolled

    def is_full(self):
        """Return True when no seats are left."""
        return self.seats_left() <= 0

    def enroll(self, n):
        """Add n students to the program. Returns nothing."""
        self.enrolled += n

    def __repr__(self):
        """Return a short description, written like the code that creates the program."""
        return f"Program(name='{self.name}', enrolled={self.enrolled}, capacity={self.capacity})"

m = Program("Measurement", 24)
m.enroll(10)
print(m)                 # Program(name='Measurement', enrolled=34, capacity=40)
# m.enroll(-5)           # should raise a ValueError
# m.enroll(100)          # should raise a ValueError

<details>
<summary>▶ Show solution</summary>

```python
class Program:
    """A degree program with an enrollment and a capacity."""

    def __init__(self, name, enrolled, capacity=40):
        self.name = name
        self.enrolled = enrolled
        self.capacity = capacity

    def seats_left(self):
        """Return how many seats are still open."""
        return self.capacity - self.enrolled

    def is_full(self):
        """Return True when no seats are left."""
        return self.seats_left() <= 0

    def enroll(self, n):
        """Add n students. Raise ValueError if n is below 1 or the program would go over capacity."""
        if n < 1:                                                                    # (a)
            raise ValueError(f"n must be at least 1, got {n}")
        if n > self.seats_left():                                                    # (b) check before changing anything
            raise ValueError(f"only {self.seats_left()} seats left, cannot enroll {n}")
        self.enrolled += n

    def __repr__(self):
        """Return a short description, written like the code that creates the program."""
        return f"Program(name='{self.name}', enrolled={self.enrolled}, capacity={self.capacity})"

m = Program("Measurement", 24)
m.enroll(10)
print(m)                 # Program(name='Measurement', enrolled=34, capacity=40)
# m.enroll(-5)           # (c) ValueError: n must be at least 1, got -5
# m.enroll(100)          # (c) ValueError: only 6 seats left, cannot enroll 100
```

</details>

### 2.4 Handling errors with `try` and `except`
`raise` is the right move when the person calling your code can fix the input. Sometimes nobody can fix it on the spot. Say the TAs typed Cara's weekly quiz scores into a spreadsheet, and you read them in as text. Most entries are fine. A few are not.

In [ ]:
typed_scores = ["88", "91.5", "ninety", "", "105", "79"]    # one quiz per week, typed by different TAs

cara = Student("Cara")
# for text in typed_scores:
#     cara.add_score(float(text))

Remove the `#` from the two loop lines and run the cell. The loop stops at the third entry with `ValueError: could not convert string to float: 'ninety'`, and the entries after it are never read. Put the `#` back. To keep every entry that can be read and set the others aside, wrap the risky line in `try` and `except`.

In [ ]:
cara = Student("Cara")
problems = []                                  # a record of the entries that failed

for text in typed_scores:
    try:
        cara.add_score(float(text))            # <---- try: the line that might fail
    except ValueError as e:                    # <---- except: runs only if a ValueError was raised
        problems.append((text, str(e)))        # <---- e is the exception object, and str(e) is its message

print(cara)
for text, message in problems:
    print(repr(text), "->", message)

For each entry, Python runs the indented line under `try`. If that line raises a `ValueError`, Python skips the rest of the `try` block, runs the `except` block, and moves on to the next entry. `as e` gives the exception object a name, so you can keep its message.

One `except ValueError` caught two different problems. `float()` raised a `ValueError` for `"ninety"` and for the empty string, and our own `add_score` raised one for `105`. That works because in 2.3 we raised the same built-in exception that `float()` uses, so one class name catches both. An exception of any other class, such as a `NameError` from a typo, is not caught and still stops the cell.

Syntax:  
```python
try:
    risky_line()               # <---- the code that might fail; keep this block short
except ValueError as e:        # <---- runs only if a ValueError was raised; e holds the exception
    handle_the_problem(e)
except KeyError:               # <---- you can add one except block for each kind of exception
    handle_another_problem()
else:                          # <---- optional: runs only if nothing was raised
    carry_on()
finally:                       # <---- optional: runs at the end in every case
    clean_up()
```

You will need `else` and `finally` less often. `finally` is for clean-up that has to happen whether or not something failed, such as the `conn.close()` that ended your SQLite code in Session 06.

> ⚠️ **Name the exception you expect.** An `except:` with no exception name after it catches every exception, including the ones that point to bugs in your own code.

In [ ]:
cara = Student("Cara")
for text in typed_scores:
    try:
        cara.add_score(flaot(text))    # a typo: flaot instead of float
    except:                            # <---- catches everything, including the NameError from the typo
        pass                           # pass means "do nothing"

print(cara)                            # every entry was skipped, and nothing told you why

The typo raised a `NameError` on every entry, and the bare `except:` swallowed each one. The loop finished without a word, and Cara has no scores at all. With `except ValueError:`, the `NameError` would have stopped the cell and shown you the typo at once. I recommend that you always name the exception you expect, and keep the `try` block to the one line that can fail.

pandas can clean a whole column this way in one call. With `errors="coerce"`, `pd.to_numeric` turns every entry it cannot convert into `NaN`, the missing-value marker, instead of raising an error.

In [ ]:
typed = pd.Series(typed_scores)
print(pd.to_numeric(typed, errors="coerce"))

`pd.to_numeric` cannot know that 105 is out of range, though, so the range check is still yours to write.

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★★☆☆  **Skills:** `try` / `except`, `raise`, `float()`

A survey asked students for their age in a free-text box. The answers are in `answers`.  
(a) Write a function `parse_age(text)` that returns the age as a float. Convert the text with `float()`, which already raises a `ValueError` for text that is not a number. Raise a `ValueError` of your own, with a clear message, when the age is below 16 or above 99.  
(b) Loop over `answers`. Put each age that `parse_age` accepts into the list `ages`, and each rejected answer, together with its error message, into the list `rejected`.  
(c) Print how many answers were accepted and their mean, then print each rejected answer with its message.

In [ ]:
answers = ["24", " 31", "twenty-two", "", "29", "-3", "45 ", "120", "27", "N/A"]
ages = []
rejected = []

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
answers = ["24", " 31", "twenty-two", "", "29", "-3", "45 ", "120", "27", "N/A"]
ages = []
rejected = []

def parse_age(text):                                         # (a)
    """Return the age in text as a float. Raise ValueError if it is not a number or not between 16 and 99."""
    age = float(text)                # raises ValueError for text that is not a number; spaces around a number are fine
    if age < 16 or age > 99:
        raise ValueError(f"age must be between 16 and 99, got {age}")
    return age

for text in answers:                                         # (b)
    try:
        ages.append(parse_age(text))
    except ValueError as e:
        rejected.append((text, str(e)))

print(len(ages), "accepted, mean age", sum(ages) / len(ages))    # (c) 5 accepted, mean age 31.2
for text, message in rejected:
    print(repr(text), "->", message)
```

</details>

### Part B. Testing your code

### 2.5 `assert` and your first test
`assert` checks something you believe is true. If the condition is true, nothing happens. If it is false, Python raises an `AssertionError`, with your message if you wrote one.

In [ ]:
assert len(penguins) == 344, "the penguins file should have 344 rows"
assert penguins["species"].nunique() == 3, "there should be three species"
assert penguins["body_mass_g"].min() > 0, "every body mass should be positive"
print("All three checks passed.")

Checks like these belong right after you load or clean a dataset. If the file on GitHub changes, or a cleaning step drops rows it should keep, the notebook stops at the check, before any chart or model is built from the wrong data.

An `assert` that calls your own code is the simplest kind of test. A **test** runs a piece of code on a small input whose answer you already know, and checks that the code gets it.

In [ ]:
assert Student("Ann", [90, 80]).average() == 85
assert Student("Bob").average() is None              # no scores yet
assert Student("Cara", [55, 65]).passed() is True    # an average of exactly 60 passes
print("All three tests passed.")

Decimals need care. Cara's average below is 83.666..., with the sixes going on, so it is not equal to 83.67. On top of that, a computer stores most decimals with a tiny error, as the last line shows.

In [ ]:
cara = Student("Cara", [78, 82, 91])
print(cara.average())                       # 83.66666666666667
# assert cara.average() == 83.67            # AssertionError, because 83.666... is not 83.67
assert round(cara.average(), 2) == 83.67    # <---- compare after rounding
print(0.1 + 0.2)                            # 0.30000000000000004, not 0.3

Compare decimals after rounding them, or with a small tolerance, which is what `unittest` does in 2.6.

`assert` and `raise` both stop the program, so which one should you use? Use `raise` in code that other people call, like `add_score`, where the error is a message to whoever passed in a bad value. Use `assert` to check your own work: that a file has the rows you expect, or that a function gives the answer you worked out by hand. Python can also be told to skip every `assert`, which is one more reason never to rely on it to check input.

### 2.6 Unit tests with `unittest`
A handful of `assert` lines stops at the first failure and says nothing about the checks that passed. Python's built-in `unittest` module groups tests into a class. Each test is a method whose name starts with `test_`, and `unittest` runs all of them and reports on each one. A **unit test** checks one small unit of code, such as one method, on its own.

In [ ]:
class TestStudent(unittest.TestCase):              # <---- a test class inherits from unittest.TestCase
    """Unit tests for the Student class."""

    def test_average_of_two_scores(self):          # <---- every test method's name starts with test_
        ann = Student("Ann", [90, 80])              # <---- a fresh object inside each test
        self.assertEqual(ann.average(), 85)         # <---- passes when the two values are equal

    def test_average_with_no_scores_is_none(self):
        bob = Student("Bob")
        self.assertIsNone(bob.average())

    def test_average_to_two_decimals(self):
        cara = Student("Cara", [78, 82, 91])
        self.assertAlmostEqual(cara.average(), 83.67, places=2)    # <---- passes when the two differ by less than 0.005

    def test_add_score_appends(self):
        dan = Student("Dan", [70])
        dan.add_score(95)
        self.assertEqual(dan.scores, [70, 95])

    def test_score_above_100_raises(self):
        eve = Student("Eve")
        with self.assertRaises(ValueError):         # <---- passes only if the indented line raises a ValueError
            eve.add_score(950)

run_tests(TestStudent)

Start with the class line. `TestStudent` inherits from `unittest.TestCase`, the way `GradStudent` inherited from `Student` last week, and that is where `assertEqual`, `assertIsNone`, `assertAlmostEqual`, and `assertRaises` come from. You call them through `self`, like any method a class inherits. `run_tests()` from the Setup cell finds every method whose name starts with `test_`, runs each one, and prints one line per test, ending in `ok`, `FAIL`, or `ERROR`. A summary follows: how many tests ran, and `OK` if all of them passed.

The tests ran in alphabetical order, not in the order they were written. That is one reason to build a fresh object inside every test. No test should depend on what another test did to a shared object.

`with self.assertRaises(ValueError):` uses `with` the way `with open(...)` did in Session 03, and the indented line runs inside it. The test passes only if that line raises a `ValueError`. If the line runs without an error, the test fails, because the class let a bad value in.

| Method | Passes when |
|---|---|
| `self.assertEqual(a, b)` | `a == b` |
| `self.assertAlmostEqual(a, b, places=2)` | `a` and `b` differ by less than 0.005 |
| `self.assertTrue(x)` and `self.assertFalse(x)` | `x` is true, or false |
| `self.assertIsNone(x)` | `x is None` |
| `self.assertIn(a, b)` | `a in b` |
| `with self.assertRaises(SomeError):` | the indented code raises `SomeError` |

A passing report is short. A failing one is longer, and it is worth learning to read. `BuggyStudent` below inherits from `Student` and overrides two methods, each with a bug planted in it.

In [ ]:
class BuggyStudent(Student):
    """A Student with two planted bugs, for practice in reading test reports."""

    def average(self):
        if len(self.scores) == 0:
            return None
        return sum(self.scores) / (len(self.scores) + 1)    # bug 1: divides by one too many

    def passed(self):
        return self.average() >= self.pasing_score          # bug 2: a typo in passing_score

class TestBuggyStudent(unittest.TestCase):
    """Two tests that should find the two bugs."""

    def test_average_of_two_scores(self):
        ann = BuggyStudent("Ann", [90, 80])
        self.assertEqual(ann.average(), 85)

    def test_average_of_60_passes(self):
        cara = BuggyStudent("Cara", [55, 65])
        self.assertTrue(cara.passed())

run_tests(TestBuggyStudent)

Each problem gets a block of its own, headed `FAIL` or `ERROR` and the name of the test, so you know where to look.

- `FAIL` means an assert method found a wrong answer. The last line of the block, `AssertionError: 56.666666666666664 != 85`, shows what the code returned first and what the test expected second.
- `ERROR` means the test crashed before it could check anything. The block is an ordinary traceback, and its last line names the exception: an `AttributeError` for the misspelled attribute. In recent versions of Python, the message ends with a guess at the name you meant.

Read each block from the bottom, the way you read a traceback in 2.1.

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★☆☆  **Skills:** `unittest.TestCase`, `assertEqual`, `assertTrue`, `assertRaises`

The next cell has the `Program` class with the checks from Exercise 2. Below the class, write a test class `TestProgram` with three test methods, and run it with `run_tests(TestProgram)`.  
(a) `test_seats_left` creates `Program("Measurement", 24)` and checks that `seats_left()` returns 16.  
(b) `test_full_at_capacity` creates a program, enrolls exactly enough students to fill it, and checks that `is_full()` returns `True`.  
(c) `test_over_capacity_raises` checks that enrolling more students than there are seats raises a `ValueError`.  
(d) Break the class on purpose: in `is_full()`, change `<=` to `<` and run the cell again. Which test catches the bug, and what does its report say? Change it back afterwards.

In [ ]:
class Program:
    """A degree program with an enrollment and a capacity."""

    def __init__(self, name, enrolled, capacity=40):
        self.name = name
        self.enrolled = enrolled
        self.capacity = capacity

    def seats_left(self):
        """Return how many seats are still open."""
        return self.capacity - self.enrolled

    def is_full(self):
        """Return True when no seats are left."""
        return self.seats_left() <= 0

    def enroll(self, n):
        """Add n students. Raise ValueError if n is below 1 or the program would go over capacity."""
        if n < 1:
            raise ValueError(f"n must be at least 1, got {n}")
        if n > self.seats_left():
            raise ValueError(f"only {self.seats_left()} seats left, cannot enroll {n}")
        self.enrolled += n

    def __repr__(self):
        """Return a short description, written like the code that creates the program."""
        return f"Program(name='{self.name}', enrolled={self.enrolled}, capacity={self.capacity})"

# Your test class here, then run_tests(TestProgram)

<details>
<summary>▶ Show solution</summary>

```python
class TestProgram(unittest.TestCase):
    """Unit tests for the Program class."""

    def test_seats_left(self):                       # (a)
        m = Program("Measurement", 24)
        self.assertEqual(m.seats_left(), 16)

    def test_full_at_capacity(self):                 # (b)
        m = Program("Measurement", 24)
        m.enroll(16)
        self.assertTrue(m.is_full())

    def test_over_capacity_raises(self):             # (c)
        m = Program("Measurement", 24)
        with self.assertRaises(ValueError):
            m.enroll(17)

run_tests(TestProgram)

# (d) With < in place of <=, test_full_at_capacity fails with "AssertionError: False is not true".
#     A full program has 0 seats left, and 0 < 0 is False. A boundary test is what catches this kind of slip.
```

</details>

### 2.7 What to test, and testing first
If time runs short in class, read this section at home. Nothing later in the notebook depends on it.

Which inputs should a test try? You cannot try every possible score, and you do not need to. Prof. Suk's notes on unit testing describe a way to choose: sort the inputs into groups that should behave the same way, called **equivalence classes**, test one or two inputs from each group, and add the edges between the groups. For `add_score`, the groups are ordinary scores, numbers below 0 or above 100, and values that are not numbers at all, and the edges are 0 and 100. For `passed()`, the edge is an average of exactly 60.

| Kind of test case | For `Student` | Why |
|---|---|---|
| a typical value | `[90, 80]` averages to 85 | the everyday case works |
| a boundary | 0 and 100 are accepted, and an average of exactly 60 passes | bugs gather at the edges, where `<` and `<=` differ |
| an empty input | a student with no scores yet | the case that would divide by zero |
| bad input | `"ninety"` and `950` raise errors | the class guards itself |
| an old bug | two students do not share one list | a bug you fixed once can come back |

Keep each test small, so that a failure points at one behavior. The last row has a name of its own: a **regression test** is a test written for a bug that was already fixed, so that the bug cannot come back unnoticed. Here is last week's `scores=[]` trap, one more time.

In [ ]:
class TrapStudent:
    """The Student with the shared-list bug from Session 11, section 2.6."""

    def __init__(self, name, scores=[]):        # <---- the trap: one default list, shared by every student
        self.name = name
        self.scores = scores

    def add_score(self, score):
        self.scores.append(score)

class TestSeparateLists(unittest.TestCase):
    """A regression test for the shared-list bug."""

    def test_new_students_do_not_share_scores(self):
        dan = TrapStudent("Dan")
        eve = TrapStudent("Eve")
        dan.add_score(77)
        self.assertEqual(eve.scores, [])          # Eve should still have no scores

run_tests(TestSeparateLists)

The test fails, and the report shows Eve's list holding Dan's 77. Now replace `TrapStudent` with `Student` in the two lines of the test that create the students, and run the cell again. Today's `Student` builds a new list in `__init__`, so the test passes. Kept with the rest of your tests, it would warn you the day someone brought the trap back.

So far we wrote the code first and the tests second. In **test-driven development**, you turn that around. You write a test for one thing the code should do, run it and watch it fail, write just enough code to make it pass, and run every test again. Then you repeat with the next behavior. Writing the test first makes you decide what a correct answer is before you write any code, and the tests stay behind as a record of what the code promises. Prof. Suk's notes add one rule: do not add a new feature until all the tests pass.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/test_cycle.png" width="800" alt="Four steps in a loop: write a test for one behavior, run it and watch it fail, write just enough code to pass, and run every test. Beside the loop, the report for the letter_grade tests before the function exists (three errors) and after (OK).">
</p>

Figure source: Made with Claude.

#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★★☆☆  **Skills:** test-driven development, `if` / `elif`, `raise`

The tests come first this time. `TestLetterGrade` in the next cell describes a function `letter_grade(score)` that does not exist yet.  
(a) Run the cell as it is and read the reports. Each test ends in `ERROR`.  
(b) Write `letter_grade(score)` above the test class: 90 and above is `"A"`, 80 and above is `"B"`, 70 and above is `"C"`, and anything lower is `"F"`. Scores below 0 or above 100 raise a `ValueError`. Run the cell after each change, and stop when every test passes.  
(c) Add a test method of your own that checks a boundary the given tests skip.

In [ ]:
# Write letter_grade(score) here, above the tests.


class TestLetterGrade(unittest.TestCase):
    """Tests for letter_grade(score), written before the function."""

    def test_typical_scores(self):
        self.assertEqual(letter_grade(95), "A")
        self.assertEqual(letter_grade(85), "B")
        self.assertEqual(letter_grade(72), "C")
        self.assertEqual(letter_grade(40), "F")

    def test_boundaries(self):
        self.assertEqual(letter_grade(90), "A")
        self.assertEqual(letter_grade(89.5), "B")
        self.assertEqual(letter_grade(70), "C")
        self.assertEqual(letter_grade(0), "F")

    def test_out_of_range_raises(self):
        with self.assertRaises(ValueError):
            letter_grade(101)
        with self.assertRaises(ValueError):
            letter_grade(-1)

run_tests(TestLetterGrade)

<details>
<summary>▶ Show solution</summary>

```python
def letter_grade(score):                                   # (b)
    """Return the letter grade for a score from 0 to 100."""
    if score < 0 or score > 100:
        raise ValueError(f"score must be between 0 and 100, got {score}")
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    else:
        return "F"


class TestLetterGrade(unittest.TestCase):
    """Tests for letter_grade(score), written before the function."""

    def test_typical_scores(self):
        self.assertEqual(letter_grade(95), "A")
        self.assertEqual(letter_grade(85), "B")
        self.assertEqual(letter_grade(72), "C")
        self.assertEqual(letter_grade(40), "F")

    def test_boundaries(self):
        self.assertEqual(letter_grade(90), "A")
        self.assertEqual(letter_grade(89.5), "B")
        self.assertEqual(letter_grade(70), "C")
        self.assertEqual(letter_grade(0), "F")

    def test_out_of_range_raises(self):
        with self.assertRaises(ValueError):
            letter_grade(101)
        with self.assertRaises(ValueError):
            letter_grade(-1)

    def test_more_boundaries(self):                        # (c) the edges of B and the top of the scale
        self.assertEqual(letter_grade(80), "B")
        self.assertEqual(letter_grade(100), "A")

run_tests(TestLetterGrade)
```

</details>

> 💡 **Tests outside a notebook.** In a project made of `.py` files, tests usually live in a file of their own, such as `test_student.py`, next to the code they test. In the terminal (Session 09), `python -m unittest` finds every file whose name starts with `test` and runs the tests in it. You may also see tests at the bottom of a file, under `if __name__ == "__main__":` followed by `unittest.main()`, which runs them only when the file itself is run, and not when another file imports it. Many projects use a library called pytest instead, which runs the same kind of tests written with plain `assert` lines. In a notebook, `run_tests()` is all you need. The folder [`Week_12/testing_demo`](https://github.com/cgpan/hudm5001/tree/main/Week_12/testing_demo) in the course repo has a small working example of all three, and its README walks through them.

### Part C. Showing your data

### 2.8 A figure is an object
Most Python charts are drawn by **matplotlib**, whose plotting module the Setup cell imported as `plt`. The idea behind it is last week's idea: a chart is an object with attributes and methods. One line creates two objects.

In [ ]:
fig, ax = plt.subplots()     # <---- one call, two objects
print(type(fig))             # the Figure: the whole image
print(type(ax))              # the Axes: one chart inside the image

`plt.subplots()` returns a `Figure` and an `Axes`, and the line unpacks them into two names, the way `a, b = 3, 4` did in Session 02. The **Figure** is the whole image, the thing you save to a file. The **Axes** is one chart inside it: the plotting area with its two axes, its tick marks, and its labels. (The name confuses everyone at first. An `Axes` is a whole chart, and it contains two axis objects, the x axis and the y axis.) The empty frame under the cell is that `Axes`, waiting for data. You draw and label by calling methods of `ax`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))                                    # width and height in inches
ax.hist(penguins["body_mass_g"].dropna(), bins=20, edgecolor="white")    # <---- draw: a method of ax
ax.set_title("Body mass of 342 penguins")                                # <---- label: more methods of ax
ax.set_xlabel("Body mass (g)")
ax.set_ylabel("Number of penguins")
plt.show()

This chart is a **histogram**. Two penguins have no measurements, so `.dropna()` leaves 342, and the title says so. `bins=20` cuts the range of body masses into 20 intervals, and each bar counts the penguins in one interval. `plt.show()` displays the figure. Without it, the notebook would also print the value that the last line returned, a `Text` object you do not need.

Here are the names of the parts of a chart.

| Word | What it is | How you reach it |
|---|---|---|
| Figure | the whole image, holding one or more charts | `fig`, from `fig, ax = plt.subplots()` |
| Axes | one chart: the plotting area, its axes, and its labels | `ax` |
| x axis and y axis | the horizontal and vertical number lines of one `Axes` | `ax.set_xlabel()`, `ax.set_xlim()` |
| title and axis labels | the text that says what the chart shows | `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` |
| legend | the key that matches colors to groups | `ax.legend()` |
| tick | a mark with a number along an axis | set for you; `ax.set_xticks()` changes them |

The picture below labels the same parts on the scatter plot you will make in 2.9, with the method that sets each one.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/fig_and_ax.png" width="800" alt="The scatter plot of flipper length and body mass. The whole image is labeled fig, the Figure, saved with fig.savefig(). The chart, with its title and labels, is labeled ax, the Axes. The title, the dots, the axis labels, and the legend are each labeled with the ax method that sets them.">
</p>

Figure source: Made with Claude.

You will also meet shorter code, online and from AI tools, that never mentions `fig` or `ax`:

```python
plt.hist(penguins["body_mass_g"].dropna(), bins=20)
plt.title("Body mass of 342 penguins")
plt.xlabel("Body mass (g)")
plt.show()
```

Each `plt.` function works on the current `Axes`, usually the last one created, so this draws the same chart. It is fine for a quick look. I recommend `fig, ax` for any chart you keep: once a figure holds two charts, you have to say which one you mean, and `fig` is the object you save. pandas offers a third route. `penguins["body_mass_g"].plot.hist(bins=20)` draws the same histogram with matplotlib underneath and returns the `Axes`.

A chart has more parts than the six in the table, and each part has a name. You will rarely need to remember the code that changes a part, because an AI tool or the matplotlib documentation can supply it. You do need the name. A request like "remove the top and right spines" or "make the tick labels larger" points at exactly one part of the chart.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/chart_anatomy.png" width="800" alt="A scatter plot of flipper length and body mass with sixteen parts named: figure, title, plot area, x axis, y axis, axis label, tick label, major tick, minor tick, gridline, spine, marker, trend line, annotation, legend, and legend title.">
</p>

Figure source: Made with Claude.

### 2.9 Four charts for four questions
Most first questions about one or two columns lead to one of four charts. Each example below starts with its question, because the question decides the chart.

The histogram in 2.8 answered the first question: how is one number spread out? The tallest bars sit between about 3,300 and 4,000 g, and to the right of about 4,000 g the bars form a long, lumpy tail that reaches past 6,000 g. The boxplot below shows who is in that tail.

The second question is how many penguins are in each group. A **bar chart** draws one bar per category, and the height of each bar is its count.

In [ ]:
counts = penguins["species"].value_counts()          # one count per species, largest first

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values)                  # <---- bar chart: one bar per category
ax.set_title("Adelie is the most common species in the sample")
ax.set_xlabel("Species")
ax.set_ylabel("Number of penguins")
plt.show()

`value_counts()` sorts the counts from largest to smallest, so the bars come out in that order too. Sorted bars are easier to compare than bars in alphabetical order.

The third question is how a number differs between groups. A **boxplot** sums up each group in one box. The line inside the box is the median, and the box holds the middle half of the group. The whiskers reach out to the farthest values within one and a half box lengths, and dots beyond them mark unusual values.

In [ ]:
species_names = ["Adelie", "Chinstrap", "Gentoo"]
masses = [penguins.loc[penguins["species"] == name, "body_mass_g"].dropna()
          for name in species_names]                 # a list comprehension: one Series of masses per species

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(masses, tick_labels=species_names)        # <---- boxplot: one box per group
ax.set_title("Gentoo penguins weigh about 1.3 kg more than the other two species")
ax.set_xlabel("Species")
ax.set_ylabel("Body mass (g)")
plt.show()

`ax.boxplot()` takes a list with one group of numbers per box, and it needs the missing values removed first, which `.dropna()` does. The median Adelie and the median Chinstrap penguin both weigh about 3,700 g, and the median Gentoo about 5,000 g, so the heavy tail of the histogram is mostly Gentoo penguins.

The fourth question is how two numbers move together. A **scatter plot** draws one dot per row, with one column on each axis. When the rows fall into groups, give each group its own color.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, group in penguins.groupby("species"):
    ax.scatter(group["flipper_length_mm"], group["body_mass_g"],
               label=name, alpha=0.7)

import numpy as np                                                    # np.polyfit does the fitting
clean = penguins[["flipper_length_mm", "body_mass_g"]].dropna()       # polyfit cannot handle missing values
flipper, mass = clean["flipper_length_mm"], clean["body_mass_g"]
slope, intercept = np.polyfit(flipper, mass, deg=1)                   # <---- deg=1: the best-fitting straight line
x_line = np.array([flipper.min(), flipper.max()])                     # two points are enough to draw a line
ax.plot(x_line, slope * x_line + intercept,
        color="black", linewidth=2)                                   # no label=, so it stays out of the legend
ax.text(0.03, 0.95, f"Slope: {slope:.1f} g per mm\nr = {flipper.corr(mass):.2f}",
        transform=ax.transAxes,                                       # place by fraction of the chart: (0, 0) bottom left, (1, 1) top right
        va="top", fontsize=10)

ax.grid(True,
        axis="both",             # "x", "y", or "both"
        linestyle="--",       # "-", "--", ":", "-."
        linewidth=0.8,        # line thickness
        color="gray",         # any matplotlib color
        alpha=0.3)            # transparency: 0 = invisible, 1 = solid
ax.set_axisbelow(True)        # draw the grid behind the bars/points instead of on top


ax.set_title("Penguins with longer flippers are heavier",
             fontsize=16,                # size in points
             fontweight="bold",          # "normal", "bold", or a number from 100 to 900
             fontfamily="serif",         # "serif", "sans-serif", or "monospace"
             color="navy")               # any matplotlib color
ax.set_xlabel("Flipper length (mm)", fontsize=12, fontstyle="italic")    # "normal" or "italic"
ax.set_ylabel("Body mass (g)", fontsize=12, fontstyle="italic")

ax.tick_params(axis="both",              # "x", "y", or "both"
               labelsize=10,             # size of the numbers on the axes
               labelcolor="dimgray")     # color of the numbers

ax.legend(title="Species", loc="lower right",
          fontsize=10,                   # size of the species names
          title_fontsize=12)             # size of the legend's own title
plt.show()

A loop over `penguins.groupby("species")` hands you one pair per species: the species name, and a smaller DataFrame with only that species' rows. Each species gets its own `scatter` call, so matplotlib gives it the next color in its list, and `label=` tells `ax.legend()` what to call it. `loc="lower right"` puts the legend in the empty corner, where it hides no dots. `alpha=0.7` makes the dots a little see-through, so overlapping dots stay visible. The cloud rises from left to right, and the correlation between flipper length and body mass is 0.87.

In this scatter plot, color shows the species, and the legend says which color is which. Color is one of six ways a chart can show a column, and the picture below has all six. When you ask for a chart, say which column goes where, as in "flipper length on the x axis, body mass on the y axis, and color by species."

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/chart_mapping.png" width="800" alt="Six small scatter plots of flipper length and body mass, one for each way to show a column: x and y position, color by species, dot size by bill length, marker shape by island, transparency, and one panel per island.">
</p>

Figure source: Made with Claude.

#### ✏️ Try It Yourself — Exercise 6
**Difficulty:** ★★☆☆☆  **Skills:** choosing a chart, `fig, ax`, titles and labels

(a) Write down, in a comment, one question about the penguins that one chart can answer. For example: are male penguins heavier than female penguins? How are flipper lengths spread out? Which island has the most penguins?  
(b) Pick the chart from this section that answers your question, and make it with `fig, ax = plt.subplots()`.  
(c) Give it a title that says what the chart shows, and label both axes, with units where there are units.  
(d) In one comment, answer your question from the chart.

In [ ]:
# (a) My question:

# Your code here

bins = range(2500, 6501, 200)                                  # <---- one set of bin edges, shared by both groups

fig, ax = plt.subplots(figsize=(7, 4))
for name, group in penguins.groupby("sex"):                    # groupby skips the 11 rows with no recorded sex
    ax.hist(group["body_mass_g"].dropna(), bins=bins,
            alpha=0.6,                                         # see-through, so the overlap shows
            label=name, edgecolor="white")                     # label= gives the legend its names
ax.set_title("Male penguins are heavier than female penguins")
ax.set_xlabel("Body mass (g)")
ax.set_ylabel("Number of penguins")
ax.legend(title="Sex")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name, group in penguins.groupby("sex"):                  # groupby skips the 11 rows with no recorded sex
    group["body_mass_g"].dropna().plot.kde(ax=ax,            # <---- ax=ax draws on our chart, not a new one
                                           label=name, linewidth=2)
ax.set_xlim(2000, 7000)                                      # kde() draws a little past the data, so fix the range
ax.set_title("Male penguins are heavier than female penguins")
ax.set_xlabel("Body mass (g)")
ax.set_ylabel("Density")
ax.legend(title="Sex")
plt.show()

<details>
<summary>▶ Show solution</summary>

```python
# (a) My question: are male penguins heavier than female penguins?
sexes = ["Female", "Male"]
masses_by_sex = [penguins.loc[penguins["sex"] == s, "body_mass_g"].dropna() for s in sexes]

fig, ax = plt.subplots(figsize=(5, 4))                       # (b) a number across groups: a boxplot
ax.boxplot(masses_by_sex, tick_labels=sexes)
ax.set_title("Male penguins are heavier than female penguins")    # (c)
ax.set_xlabel("Sex")
ax.set_ylabel("Body mass (g)")
plt.show()

# (d) Yes. The median male weighs 4,300 g and the median female 3,650 g, about 650 g less.
#     Eleven penguins have no recorded sex, so they are not in the chart.

# Another question: which island has the most penguins? A bar chart answers it.
island_counts = penguins["island"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(island_counts.index, island_counts.values)
ax.set_title("Almost half of the penguins in the sample come from Biscoe")
ax.set_xlabel("Island")
ax.set_ylabel("Number of penguins")
plt.show()
# Biscoe has 168 of the 344 penguins, Dream 124, and Torgersen 52.
```

</details>

### 2.10 Several charts in one figure, and saving for your report
A figure can hold several charts. Give `plt.subplots()` a number of rows and columns, and it returns an array of `Axes`, one per chart, in place of a single `ax`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)    # <---- 1 row of 3 charts, on the same scales
for ax, name in zip(axes, species_names):                                    # <---- pair each chart with one species
    group = penguins[penguins["species"] == name]
    ax.hist(group["body_mass_g"].dropna(), bins=15, edgecolor="white")
    ax.set_title(name)
    ax.set_xlabel("Body mass (g)")
axes[0].set_ylabel("Number of penguins")       # the y label only on the first chart
fig.suptitle("Body mass by species")           # <---- a title for the whole figure
fig.tight_layout()                             # <---- make room so that labels do not overlap
plt.show()

`zip()` from Session 07 pairs the first chart with the first species, and so on. `sharex=True` and `sharey=True` put all three charts on the same scales. Without them, each chart picks its own range, and the three piles of bars line up across the figure even though the Gentoo pile sits about 1.3 kg to the right. Try it: delete the two arguments and run the cell again.

To put a chart in your report, save its figure to a file with `fig.savefig()`. Save before `plt.show()`, while the figure is still open.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(masses, tick_labels=species_names)
ax.set_title("Gentoo penguins weigh about 1.3 kg more than the other two species")
ax.set_xlabel("Species")
ax.set_ylabel("Body mass (g)")
fig.savefig("body_mass_by_species.png", dpi=300, bbox_inches="tight")    # <---- 300 dots per inch, empty margins trimmed
plt.show()

In Colab, the file appears in the Files panel, behind the folder icon on the left. Right-click it and choose Download. Colab deletes these files when the session ends, so download them before you close the notebook. A PNG saved at 300 dpi stays sharp in a Word document or a PDF, and `bbox_inches="tight"` trims the empty margin around the chart and keeps long labels from being cut off. Open the saved file once to check it, since a figure can look a little different on disk than in the notebook.  

### 2.11 Interactive charts, and choosing a chart
If time runs short in class, read the interactive part at home and pick up at the table of charts. Nothing later depends on it.

matplotlib makes still images, which is what a report needs. When you explore data on screen, an interactive chart is handy: you can hover over a dot to see its values, zoom into a corner, and click a group in the legend to hide it. **Plotly Express** makes one in a single line from a DataFrame.

In [ ]:
import plotly.express as px     # Plotly Express, preinstalled in Colab; elsewhere, run pip install plotly first

fig = px.scatter(penguins, x="bill_length_mm", y="bill_depth_mm", color="species",
                 hover_data=["island", "sex"],
                 title="Bill length and bill depth of 342 penguins")
fig.show()

Hover over a dot, drag a box to zoom in, and double-click to zoom back out. Click a species in the legend to hide it, and double-click one to show it alone. `color="species"` does the grouping that took a `groupby` loop in 2.9. This `fig` is a Plotly figure, an object of a different class from matplotlib's `Figure`, with methods of its own. `fig.write_html("bills.html")` saves it as a web page that keeps the hovering and zooming, which is handy for sharing but no use in a printed report.

This chart also carries a warning. Show one species at a time in the legend, or look at the numbers.

In [ ]:
print("all penguins:", round(penguins["bill_length_mm"].corr(penguins["bill_depth_mm"]), 2))
for name, group in penguins.groupby("species"):
    print(name, round(group["bill_length_mm"].corr(group["bill_depth_mm"]), 2))

Across all the penguins, the correlation is negative: longer bills seem to go with shallower bills. Within every species, it is positive. The overall slope comes from mixing the groups. Gentoo penguins have long but shallow bills, so they sit in the lower right of the chart and pull the overall trend down. A reversal like this is called **Simpson's paradox**. Coloring a chart by group is the quickest way to catch one, so look at the groups before you trust a single correlation.

Here is Part C in one table.

| Your question | Chart | matplotlib |
|---|---|---|
| How is one number spread out? | histogram | `ax.hist(x, bins=20)` |
| How many are in each group? | bar chart | `ax.bar(names, counts)` |
| How does a number differ between groups? | boxplot, or one histogram per group | `ax.boxplot(list_of_groups, tick_labels=names)` |
| How do two numbers move together? | scatter plot, colored by group if there are groups | `ax.scatter(x, y)` |
| How does a number change over time, or along an ordered x? | line chart | `ax.plot(x, y)` |

We did not draw a line chart today. You saw line charts in Assignment 07, where the network's accuracy on the training and validation students changed epoch by epoch. Two other libraries come up often: seaborn, which draws grouped charts like ours with a `hue=` argument, and plotnine, which brings R's ggplot2 grammar to Python if you know R. Both are installed in Colab.

The five charts in the table answer most first questions. Many other charts have names of their own, and the two pictures below sort the common ones by the question they answer, from density plots to the charts that show how well a model does. When one fits your question, ask for it by name. If you can picture a chart but do not know what it is called, describe it to an AI tool and ask.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/chart_types_1.png" width="800" alt="Twelve small charts, named and grouped by the question they answer. How is one number spread out: histogram, density plot, box plot, violin plot. How do groups compare: bar chart, grouped bar chart, stacked bar chart, error bar plot. How do two numbers move together: scatter plot, trend line, bubble chart, hexbin plot.">
</p>

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/chart_types_2.png" width="800" alt="Eleven more named charts. Change over time: line chart, stacked area chart. Shares of a whole: pie chart, 100% stacked bar chart. Many numbers at once: heatmap, pair plot, small multiples. Model results: training curve, confusion matrix, ROC curve, coefficient plot. Also a list of other chart names.">
</p>

Figure source: Made with Claude.

Your final project report needs at least two figures. Before a figure goes in, check it against this list.

- The title says what the reader should see, as in "Penguins with longer flippers are heavier". A title like "Flipper length and body mass" only names the variables.
- Both axes have labels, with units: `Body mass (g)`, not `body_mass_g`.
- Bars start at zero. `ax.bar` does this for you, so leave it that way, since the length of a bar is its value.
- The caption says how many cases the figure shows and which were left out, as in "342 of 344 penguins; two have no measurements."
- The figure still makes sense in black and white, and to a reader who cannot tell red from green. Different marker shapes, or labels written next to the groups, help.
- Prefer a bar chart to a pie chart, and avoid 3D effects, since lengths are easier to compare than angles or volumes.
- The file is a PNG saved with `dpi=300` and `bbox_inches="tight"`, and you opened it once to check it.
- In an APA-style report, each figure gets a number in bold and a title in italics above it, and a note below it when the reader needs one.

#### ✏️ Try It Yourself — Exercise 7
**Difficulty:** ★★★★☆  **Skills:** `raise`, `unittest`, `fig, ax`, a scatter plot colored by group, `savefig`

This exercise puts today's three parts into one function. The next cell has a skeleton of `plot_by_group(df, x, y, group)`, which draws a scatter plot of two columns with one color per group.  
(a) Before drawing anything, the function should raise a `KeyError` that names the missing column when `x`, `y`, or `group` is not a column of `df`. A loop over the three names keeps this short.  
(b) Finish the drawing: one `ax.scatter` call per group, axis labels from the column names, and a legend. The function returns the figure and the `Axes`.  
(c) Write a test class with two tests. One checks that a misspelled column raises a `KeyError`. The other checks that the returned `Axes` has the right x label; `ax.get_xlabel()` returns it as a string. End that test with `plt.close(fig)`, so the test's chart is not displayed.  
(d) Call your function on the penguins with two number columns and a group column of your choice, give the chart a title that says what it shows, and save it as a PNG with `dpi=300`.

In [ ]:
def plot_by_group(df, x, y, group):
    """Draw a scatter plot of column y against column x, with one color per value of column group.
    Raise KeyError if a column is missing. Return the figure and the Axes."""
    # (a) check the three column names here

    fig, ax = plt.subplots(figsize=(7, 5))
    # (b) one ax.scatter call per group, then the axis labels and the legend

    return fig, ax


# (c) your test class here


# (d) your chart here

<details>
<summary>▶ Show solution</summary>

```python
def plot_by_group(df, x, y, group):
    """Draw a scatter plot of column y against column x, with one color per value of column group.
    Raise KeyError if a column is missing. Return the figure and the Axes."""
    for column in [x, y, group]:                                       # (a)
        if column not in df.columns:
            raise KeyError(f"no column named '{column}'; check the spelling")

    fig, ax = plt.subplots(figsize=(7, 5))
    for name, rows in df.groupby(group):                               # (b)
        ax.scatter(rows[x], rows[y], label=name, alpha=0.7)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(title=group)
    return fig, ax


class TestPlotByGroup(unittest.TestCase):                              # (c)
    """Tests for plot_by_group."""

    def test_missing_column_raises(self):
        with self.assertRaises(KeyError):
            plot_by_group(penguins, "bill_lenght_mm", "bill_depth_mm", "species")

    def test_x_label_is_the_column_name(self):
        fig, ax = plot_by_group(penguins, "bill_length_mm", "bill_depth_mm", "species")
        self.assertEqual(ax.get_xlabel(), "bill_length_mm")
        plt.close(fig)                       # the test does not need to show its chart

run_tests(TestPlotByGroup)

fig, ax = plot_by_group(penguins, "flipper_length_mm", "body_mass_g", "island")    # (d)
ax.set_title("Penguins with longer flippers are heavier on every island")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Body mass (g)")
fig.savefig("flipper_mass_by_island.png", dpi=300, bbox_inches="tight")
plt.show()
```

</details>

### 2.12 Asking AI for a chart, and checking the result
From here on, most of your charts will probably start as a request to an AI tool. The tool writes the code in seconds, and it fills in whatever your request leaves out. The names from this part of the session, for chart types, chart parts, and the ways to show a column, let you leave less out, and they let you say exactly what to change when the first chart misses. Memorizing matplotlib's arguments matters much less. When you need one, AI or the documentation can tell you.

Here is a vague request next to a precise one, both about the penguins. The vague request leaves the tool to guess. The precise one names six things: the data, the chart type, which column goes where, the text on the chart, the layout, and the tool.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/ai_chart_request.png" width="800" alt="A vague request, make a chart of the penguins data, answered with a histogram of every number column, next to a precise request with its parts highlighted (the data, the chart type, which column goes where, the text on the chart, the layout, the tool) and the scatter plot it produces.">
</p>

Figure source: Made with Claude.

The first chart is rarely the last. The picture below takes the chart from the precise request through three more rounds, and each request names the part it changes.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/ai_chart_rounds.png" width="800" alt="The scatter plot refined in three rounds: colors that colorblind readers can tell apart, no top or right spine with light horizontal gridlines, and a trend line for each species. Four tips: name the part, change one or two things, send your current code, show the chart.">
</p>

Figure source: Made with Claude.

Code from an AI tool can run without an error and still draw a misleading chart, the way Bob's score of 950 went in without an error in 2.1. Check every chart before you use it, with the questions below and the final-project list in 2.11. If you turn a chart into a function you will reuse, test it, as in Exercise 7.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/chart_checks.png" width="800" alt="Two bar charts of mean body mass by species. With the y axis starting at 3,500 g, the Gentoo bar looks about eight times as tall as the Adelie bar. Starting at 0, the bars show that Gentoo penguins are about 37% heavier. Beside them, five checks: the rows, the numbers, the axes, the colors, and the question.">
</p>

Figure source: Made with Claude.

---
## 3. Conclusions & Key Takeaways
- When Python cannot run a line, it raises an exception and prints a traceback. Read it from the bottom: the last line names the exception and gives a message, and the arrows above it show where Python stopped. That line can be far from the real mistake.
- The exception's name tells you what to look for: `NameError`, `KeyError`, `TypeError`, `AttributeError`, `ValueError`, `IndexError`, `ZeroDivisionError`, or `FileNotFoundError`. Every one of them is a class that inherits from `Exception`.
- Python raises errors only for lines it cannot run. A value that runs fine but makes no sense, like a score of 950, needs a check of your own: `raise ValueError("what was expected, and what arrived")`.
- `try` and `except` keep a program running when some input is bad. Name the exception you expect and keep the `try` block short. A bare `except:` also hides your own typos.
- `assert condition, "message"` checks something you believe, such as the number of rows after loading a file. Use `raise` for input that other people send in.
- A unit test runs a small piece of code on an input whose answer you know. In `unittest`, tests are methods whose names start with `test_`, inside a class that inherits from `unittest.TestCase`, and `assertEqual`, `assertAlmostEqual`, and `assertRaises` do the checking. `FAIL` means a wrong answer, and `ERROR` means the test crashed.
- Good test cases cover a typical value, the boundaries, an empty input, bad input, and every bug you have fixed before. In test-driven development, you write the test first, watch it fail, and then write the code.
- `fig, ax = plt.subplots()` creates a `Figure`, the whole image, and an `Axes`, one chart inside it. You draw and label with methods of `ax`, and you save with `fig.savefig(..., dpi=300, bbox_inches="tight")`.
- Let the question pick the chart: a histogram for one number, a bar chart for group sizes, a boxplot for a number across groups, a scatter plot for two numbers, and a line chart for change over time.
- Color by group before you trust a pattern. Bill length and bill depth slope down across all the penguins, but up within every species.
- Ask AI for a chart by name, say which column goes where, and refine it in rounds by naming the parts to change. Then check it: the rows that went in, one number you computed yourself, the axes, and the colors.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| raise your own error | `raise ValueError(f"score must be between 0 and 100, got {score}")` |
| check the type of a value | `isinstance(score, (int, float))` |
| handle an expected error | `try: ... except ValueError as e: print(e)` |
| clean-up that always runs | `finally: conn.close()` |
| text to numbers, bad entries to `NaN` | `pd.to_numeric(s, errors="coerce")` |
| check an assumption | `assert len(df) == 344, "message"` |
| a test class | `class TestStudent(unittest.TestCase):` with methods named `test_...` |
| check a value | `self.assertEqual(actual, expected)` |
| check a decimal | `self.assertAlmostEqual(actual, expected, places=2)` |
| check that an error is raised | `with self.assertRaises(ValueError):` |
| run the tests in this notebook | `run_tests(TestStudent)` |
| a figure with one chart | `fig, ax = plt.subplots(figsize=(7, 4))` |
| a figure with three charts side by side | `fig, axes = plt.subplots(1, 3, sharey=True)` |
| histogram, bar, box, scatter, line | `ax.hist(x)`, `ax.bar(names, counts)`, `ax.boxplot(groups)`, `ax.scatter(x, y)`, `ax.plot(x, y)` |
| title and axis labels | `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` |
| legend | `label=` in each drawing call, then `ax.legend()` |
| save for a report | `fig.savefig("name.png", dpi=300, bbox_inches="tight")` |
| interactive chart | `px.scatter(df, x="...", y="...", color="...")` |

### ⏭️ Coming Up Next
**Coding practices with AI.** You have used AI tools on the assignments all semester, under the course's AI policy. Next week is about using them well: asking for code you can check, reading what comes back, and testing it before you trust it. The tests from Part B carry over directly, since a test checks AI-written code the same way it checks yours. After that, the semester turns to your final projects.

### 📌 This Week's Assignment
I will post this week's assignment on Canvas and in the `Assignments/` folder of the course GitHub.

> 💡 **Tip:** When a cell fails, read the last line of the traceback first. The exception's name and its message usually tell you what to fix, and the arrow just above shows where Python stopped. If you ask an AI tool for help, paste that last line together with your code, since the message holds most of what the tool needs.